# HBCC Standard KD và HBCC-DKD trên Kaggle

Notebook này huấn luyện **HBCC-Small** và **HBCC-Medium** bằng hai phương pháp:

1. `standard`: KD chuẩn với `KL(teacher || student)`.
2. `dkd`: Decoupled Knowledge Distillation với TCKD và NCKD.

Teacher là checkpoint ResNet-18 baseline đã huấn luyện bằng CE. Toàn bộ dữ liệu train/validation/test chỉ dùng `ToTensor + Normalize`; notebook và runner sẽ từ chối teacher hoặc recipe có augmentation. HBCC student luôn được khởi tạo lại từ cùng seed, không warm-start từ checkpoint HBCC-CE.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch
import yaml

## 1. Cấu hình đường dẫn và thực nghiệm

Các đường dẫn quan trọng đều nằm trong ô dưới đây. Trên Kaggle:

- `REPO_ROOT`: thư mục chứa repository. Có thể để `None` nếu notebook nằm trong repository hoặc repo ở `/kaggle/working/Lightweight-Context-Cluster`.
- `DATA_ROOTS`: thư mục mà `torchvision.datasets.CIFAR10/CIFAR100` sử dụng làm `root`.
- `TEACHER_CHECKPOINTS`: mỗi dataset/seed phải trỏ đến `best.pth` của ResNet-18 no-augmentation.
- `TEACHER_CONFIGS`: có thể để `None`; config sẽ được lấy trực tiếp từ checkpoint.
- `REFERENCE_RESULTS_ROOTS`: tùy chọn, dùng để đọc kết quả CE cũ vào bảng so sánh; không dùng để huấn luyện KD.
- `/kaggle/input` là read-only, vì vậy `OUTPUT_ROOT` nên đặt trong `/kaggle/working`.

In [ ]:
# ---------- Duong dan Kaggle can chinh ----------
REPO_ROOT = None  # Vi du: Path('/kaggle/input/lightweight-context-cluster')

DATASETS = ['cifar10']  # Co the dung ['cifar10', 'cifar100']
SEEDS = [42]
DATA_ROOTS = {
    'cifar10': Path('/kaggle/working/data'),
    'cifar100': Path('/kaggle/working/data'),
}
OUTPUT_ROOT = Path('/kaggle/working/hbcc_kd_dkd_runs')

TEACHER_CHECKPOINTS = {
    ('cifar10', 42): Path('/kaggle/input/CHANGE-ME/cifar10_resnet18/best.pth'),
    # ('cifar100', 42): Path('/kaggle/input/CHANGE-ME/cifar100_resnet18/best.pth'),
}
TEACHER_CONFIGS = {
    ('cifar10', 42): None,  # Hoac Path('/kaggle/input/.../config.yaml')
    # ('cifar100', 42): None,
}
REFERENCE_RESULTS_ROOTS = [
    # Path('/kaggle/input/CHANGE-ME/baseline-runs'),
]


STUDENTS = ['hbcc_small', 'hbcc_medium']
METHODS = ['standard', 'dkd']
KD_EPOCHS = 200
EXPECTED_TEACHER_EPOCHS = 200  # Dat None neu khong muon rang buoc epoch teacher
LABEL_SMOOTHING = 0.10  # Giu giong CE baseline
KD_TEMPERATURE = 4.0
STANDARD_KD_ALPHA = 0.5
DKD_TCKD_WEIGHT = 1.0
DKD_NCKD_WEIGHT = 4.0
DKD_WARMUP_EPOCHS = 20

DOWNLOAD_DATA = True  # False neu CIFAR da co san trong DATA_ROOTS
NUM_WORKERS = 4
DEVICE = 'auto'
PRINT_EVERY = 5
SHOW_PROGRESS = False
FORCE = False
SMOKE = False  # True: FakeData, 1 epoch va 1 batch de kiem tra luong

In [ ]:
def find_repo_root(explicit_root=None):
    candidates = []
    if explicit_root is not None:
        candidates.append(Path(explicit_root))
    candidates.extend([
        Path.cwd(),
        Path.cwd().parent,
        Path('/kaggle/working/Lightweight-Context-Cluster'),
    ])
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'tools' / 'run_kd_comparison.py').is_file():
            return candidate
    raise FileNotFoundError('Khong tim thay repository chua tools/run_kd_comparison.py')


ROOT = find_repo_root(REPO_ROOT)
RUNNER = ROOT / 'tools' / 'run_kd_comparison.py'
OUTPUT_ROOT = OUTPUT_ROOT.expanduser().resolve()

assert DATASETS and set(DATASETS) <= {'cifar10', 'cifar100'}
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert STUDENTS and set(STUDENTS) <= {'hbcc_small', 'hbcc_medium'}
assert METHODS and set(METHODS) <= {'standard', 'dkd'}
assert isinstance(KD_EPOCHS, int) and KD_EPOCHS > 0
assert KD_TEMPERATURE > 0 and 0 < STANDARD_KD_ALPHA <= 1
assert DKD_TCKD_WEIGHT >= 0 and DKD_NCKD_WEIGHT >= 0
assert DKD_TCKD_WEIGHT + DKD_NCKD_WEIGHT > 0
assert isinstance(DKD_WARMUP_EPOCHS, int) and DKD_WARMUP_EPOCHS >= 0
assert 0 <= LABEL_SMOOTHING < 1

required_keys = {(dataset, seed) for dataset in DATASETS for seed in SEEDS}
missing_checkpoints = sorted(required_keys - set(TEACHER_CHECKPOINTS))
if missing_checkpoints:
    raise KeyError(f'Thieu TEACHER_CHECKPOINTS cho: {missing_checkpoints}')
for key in sorted(required_keys):
    checkpoint = Path(TEACHER_CHECKPOINTS[key]).expanduser()
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Khong tim thay teacher checkpoint cho {key}: {checkpoint}')
    teacher_config = TEACHER_CONFIGS.get(key)
    if teacher_config is not None and not Path(teacher_config).expanduser().is_file():
        raise FileNotFoundError(f'Khong tim thay teacher config cho {key}: {teacher_config}')

print('Repository  :', ROOT)
print('Python      :', sys.executable)
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))
print('Output root :', OUTPUT_ROOT)
print('Datasets    :', DATASETS)
print('Seeds       :', SEEDS)
print('Students    :', STUDENTS)
print('Methods     :', METHODS)
print('KD epochs   :', KD_EPOCHS)
print('Smoke       :', SMOKE)

## 2. Kiểm tra checkpoint và giao thức no-augmentation

Preflight tải checkpoint trên CPU, kiểm tra state dict, config nhúng, kiến trúc ResNet-18, dataset, seed, epoch và `augmentation: none`. Nếu `TEACHER_CONFIGS[key] = None`, runner tự tạo một bản config trong `OUTPUT_ROOT/_teacher_configs`.

In [ ]:
def build_command(dataset, seed, validate_only=False):
    key = (dataset, seed)
    command = [
        sys.executable, str(RUNNER),
        '--dataset', dataset,
        '--seed', str(seed),
        '--students', *STUDENTS,
        '--methods', *METHODS,
        '--teacher-checkpoint', str(Path(TEACHER_CHECKPOINTS[key]).expanduser().resolve()),
        '--data-root', str(Path(DATA_ROOTS[dataset]).expanduser().resolve()),
        '--output', str(OUTPUT_ROOT),
        '--python', sys.executable,
        '--device', DEVICE,
        '--epochs', str(KD_EPOCHS),
        '--temperature', str(KD_TEMPERATURE),
        '--standard-alpha', str(STANDARD_KD_ALPHA),
        '--dkd-tckd-weight', str(DKD_TCKD_WEIGHT),
        '--dkd-nckd-weight', str(DKD_NCKD_WEIGHT),
        '--dkd-warmup-epochs', str(DKD_WARMUP_EPOCHS),
        '--label-smoothing', str(LABEL_SMOOTHING),
        '--workers', str(NUM_WORKERS),
        '--print-every', str(PRINT_EVERY),
        '--download-data' if DOWNLOAD_DATA else '--no-download-data',
    ]
    teacher_config = TEACHER_CONFIGS.get(key)
    if teacher_config is not None:
        command.extend(['--teacher-config', str(Path(teacher_config).expanduser().resolve())])
    if EXPECTED_TEACHER_EPOCHS is not None:
        command.extend(['--expected-teacher-epochs', str(EXPECTED_TEACHER_EPOCHS)])
    if SHOW_PROGRESS:
        command.append('--progress')
    if SMOKE:
        command.append('--smoke')
    if FORCE:
        command.append('--force')
    if validate_only:
        command.append('--validate-only')
    return command


def run_command(command):
    print('\n>', subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


for dataset in DATASETS:
    for seed in SEEDS:
        run_command(build_command(dataset, seed, validate_only=True))

## 3. Chạy Standard KD và DKD

Mỗi dataset/seed tạo bốn run theo thứ tự: hai HBCC với Standard KD và hai HBCC với DKD. Run hoàn tất và có config tương thích sẽ được bỏ qua; `FORCE=True` cho phép chạy lại đúng tên run.

In [ ]:
for dataset in DATASETS:
    for seed in SEEDS:
        run_command(build_command(dataset, seed, validate_only=False))

## 4. Tổng hợp kết quả

Bảng chính đọc kết quả Standard KD/DKD vừa chạy. Nếu thêm thư mục vào `REFERENCE_RESULTS_ROOTS`, bảng cũng đọc các run CE no-augmentation trước đó để đối chiếu; các checkpoint CE chỉ được đọc metadata, không được nạp vào student KD.

In [ ]:
records = []
search_roots = [OUTPUT_ROOT, *[Path(path).expanduser().resolve() for path in REFERENCE_RESULTS_ROOTS]]
seen_metrics = set()
for search_root in search_roots:
    if not search_root.exists():
        print('Bo qua reference root khong ton tai:', search_root)
        continue
    for metrics_path in sorted(search_root.rglob('test_metrics.json')):
        metrics_path = metrics_path.resolve()
        if metrics_path in seen_metrics:
            continue
        seen_metrics.add(metrics_path)
        config_path = metrics_path.parent / 'config.yaml'
        if not config_path.is_file():
            continue
        cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        protocol = cfg.get('protocol', {})
        train_cfg = cfg.get('train', {})
        experiment = cfg.get('experiment', {})
        distillation = cfg.get('distillation', {})
        dataset = protocol.get('dataset')
        seed = int(train_cfg.get('seed', -1))
        model = experiment.get('model_key', cfg.get('model', {}).get('name'))
        augmentation = protocol.get('augmentation')
        method = distillation.get('method', train_cfg.get('kd_method', 'none'))
        method = 'ce' if method in {None, 'none', 'ce'} else method
        expected_epochs = 1 if SMOKE else KD_EPOCHS
        run_epochs = int(train_cfg.get('epochs', -1))
        if dataset not in DATASETS or seed not in SEEDS or augmentation != 'none':
            continue
        if run_epochs != expected_epochs:
            continue
        if method not in {'ce', 'standard', 'dkd'}:
            continue
        if method == 'ce' and model not in {*STUDENTS, 'resnet18'}:
            continue
        if method in {'standard', 'dkd'}:
            if model not in STUDENTS:
                continue
            expected_teacher = Path(TEACHER_CHECKPOINTS[(dataset, seed)]).expanduser().resolve()
            actual_teacher = Path(distillation.get('teacher_checkpoint', '')).expanduser().resolve()
            if actual_teacher != expected_teacher:
                continue
            if float(distillation.get('temperature', -1)) != float(KD_TEMPERATURE):
                continue
            if float(train_cfg.get('label_smoothing', -1)) != float(LABEL_SMOOTHING):
                continue
        if method == 'standard' and float(distillation.get('alpha', -1)) != float(STANDARD_KD_ALPHA):
            continue
        if method == 'dkd':
            dkd_matches = (
                float(distillation.get('dkd_tckd_weight', -1)) == float(DKD_TCKD_WEIGHT)
                and float(distillation.get('dkd_nckd_weight', -1)) == float(DKD_NCKD_WEIGHT)
                and int(distillation.get('dkd_warmup_epochs', -1)) == int(DKD_WARMUP_EPOCHS)
            )
            if not dkd_matches:
                continue
        records.append({
            'dataset': dataset,
            'seed': seed,
            'model': model,
            'method': method,
            'epochs': run_epochs,
            'temperature': distillation.get('temperature'),
            'standard_alpha': distillation.get('alpha') if method == 'standard' else None,
            'tckd_weight': distillation.get('dkd_tckd_weight') if method == 'dkd' else None,
            'nckd_weight': distillation.get('dkd_nckd_weight') if method == 'dkd' else None,
            'test_acc1': float(metrics['test_acc1']),
            'test_acc5': metrics.get('test_acc5'),
            'run': metrics_path.parent.name,
            'path': str(metrics_path.parent),
        })

summary = pd.DataFrame(records)
if summary.empty:
    raise RuntimeError('Khong tim thay ket qua test phu hop')
summary = summary.sort_values(['dataset', 'seed', 'model', 'method']).reset_index(drop=True)
kd_summary = summary[summary['method'].isin(METHODS)].copy()
expected_kd_runs = len(DATASETS) * len(SEEDS) * len(STUDENTS) * len(METHODS)
if len(kd_summary) != expected_kd_runs:
    raise RuntimeError(f'Ket qua KD chua du: {len(kd_summary)}/{expected_kd_runs} run')
summary_path = OUTPUT_ROOT / 'hbcc_kd_dkd_summary.csv'
summary.to_csv(summary_path, index=False)
print('Da luu:', summary_path)
display(summary)
display(summary.pivot_table(index=['dataset', 'seed', 'model'], columns='method', values='test_acc1'))